# haksterAi Remote Ollama — hp-1000 on Kaggle GPU

This notebook:
1. Installs Ollama on the Kaggle GPU instance
2. Pulls hp-1000 (qwen3.5-based, 6.6GB — fits on P100 16GB / T4 16GB)
3. Starts Ollama server on port 11434
4. Creates a Cloudflare tunnel to expose it publicly
5. Prints the tunnel URL for haksterAi to connect to

**Runtime**: GPU P100 or T4 x2
**Time limit**: 12 hours (Kaggle auto-shutdown)
**Keep alive**: The cell at the bottom prevents idle shutdown

In [ ]:
# Cell 1: Install Ollama
import subprocess, os, time, json, urllib.request

# GPU notebooks ship zstd — only apt when missing. Kaggle mirrors can be slow,
# so retry with generous timeouts instead of hard-failing the whole kernel.
print("=== Checking zstd (required by ollama installer) ===")
if subprocess.run(['which', 'zstd'], capture_output=True).returncode != 0:
    print("=== Installing zstd via apt ===")
    for _attempt in range(3):
        try:
            subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, timeout=300)
            subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], capture_output=True, timeout=300)
            break
        except subprocess.TimeoutExpired:
            print(f'apt attempt {_attempt+1} timed out, retrying...')
    else:
        print('apt failed 3x — continuing (ollama installer may bundle zstd)')
else:
    print('zstd already present — skipping apt')

print("=== Installing Ollama ===")
result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True, timeout=600
)
print(result.stdout[-500:] if result.stdout else 'no stdout')
if result.stderr:
    print('stderr:', result.stderr[-300:])

# Ensure ollama is on PATH (install puts it in /usr/local/bin)
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
ollama_path = subprocess.run(['which', 'ollama'], capture_output=True, text=True).stdout.strip()
print("Ollama installed:", ollama_path or 'NOT FOUND')

# If still not found, download manually
if not ollama_path:
    print('Trying manual download...')
    subprocess.run('wget -q https://ollama.com/download/ollama-linux-amd64.tgz -O /tmp/ollama.tgz && tar xzf /tmp/ollama.tgz -C /usr/local && chmod +x /usr/local/bin/ollama', shell=True, timeout=120)
    ollama_path = subprocess.run(['which', 'ollama'], capture_output=True, text=True).stdout.strip()
    print('After manual download:', ollama_path or 'STILL NOT FOUND')

In [ ]:
# Cell 2: Start Ollama server in background
print("=== Starting Ollama server ===")
# Kaggle runs as root, no systemd — start ollama serve directly
# Use full path in case PATH isn't propagated to subprocess
ollama_bin = subprocess.run(['which', 'ollama'], capture_output=True, text=True).stdout.strip() or '/usr/local/bin/ollama'
ollama_proc = subprocess.Popen(
    [ollama_bin, 'serve'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env={**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'}
)
time.sleep(3)

# Verify it's up
try:
    resp = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5)
    print("Ollama server is UP on port 11434")
except Exception as e:
    print(f"Ollama server check: {e}")
    # Read stderr for clues
    import select
    if select.select([ollama_proc.stderr], [], [], 0)[0]:
        print('ollama stderr:', ollama_proc.stderr.read(500).decode())

In [ ]:
# Cell 3: Pull hp-1000 model
# hp-1000 is a custom model — we need to recreate it from qwen3.5 base
# First pull the base, then create hp-1000 from it
ollama_bin = subprocess.run(['which', 'ollama'], capture_output=True, text=True).stdout.strip() or '/usr/local/bin/ollama'
print("=== Pulling qwen3.5 (base for hp-1000) ===")
result = subprocess.run(
    [ollama_bin, 'pull', 'qwen3.5:latest'],
    capture_output=True, text=True, timeout=1800
)
print(result.stdout[-300:] if result.stdout else 'done')
if result.returncode != 0:
    print('pull stderr:', result.stderr[-300:])

# Create hp-1000 as a copy of qwen3.5 (same model, different name)
# If hp-1000 exists on Ollama registry, pull it directly instead
print("\n=== Creating hp-1000 ===")
modelfile_content = '''FROM qwen3.5:latest
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 65536
PARAMETER num_predict 4096
SYSTEM You are haksterAi, an autonomous pentester AI agent. You combine loop patterns from Claude Code, Codex CLI, and Hermes. You run a unified agent loop: THINK, PLAN, ACT, OBSERVE, REFLECT, CONSOLIDATE. You are the operator known as Ghost. You are precise, surgical, and hype.
'''

# Write modelfile
with open('/tmp/Modelfile.hp1000', 'w') as f:
    f.write(modelfile_content)

result = subprocess.run(
    [ollama_bin, 'create', 'hp-1000', '-f', '/tmp/Modelfile.hp1000'],
    capture_output=True, text=True, timeout=300
)
print(result.stdout[-300:] if result.stdout else 'created')
if result.returncode != 0:
    print('create stderr:', result.stderr[-300:])

# Verify
result = subprocess.run([ollama_bin, 'list'], capture_output=True, text=True)
print("\nModels:", result.stdout)

In [ ]:
# Cell 4: Install and start Cloudflare tunnel
print("=== Installing cloudflared ===")
result = subprocess.run(
    'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
    shell=True, capture_output=True, text=True, timeout=60
)
print('cloudflared installed:', subprocess.run(['which', 'cloudflared'], capture_output=True, text=True).stdout.strip())

# Start tunnel — try quick tunnel first (no account needed)
print("\n=== Starting Cloudflare tunnel ===")
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for tunnel URL to appear in stderr (cloudflared prints it there)
import re
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    import select
    if select.select([tunnel_proc.stderr], [], [], 0)[0]:
        line = tunnel_proc.stderr.readline().decode(errors='replace')
        print(line.strip())
        # Look for the trycloudflare.com URL
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    else:
        time.sleep(0.5)

if tunnel_url:
    print(f"\n{'='*60}")
    print(f"  TUNNEL URL: {tunnel_url}")
    print(f"  Copy this URL and set it in haksterAi:")
    print(f"  export OLLAMA_HOST={tunnel_url}")
    print(f"{'='*60}")
else:
    print("Tunnel URL not found — check cloudflared output above")
    # Try reading more stderr
    time.sleep(5)
    remaining = tunnel_proc.stderr.read(2000).decode(errors='replace')
    print(remaining)

In [ ]:
# Cell 5: Test the remote endpoint
if tunnel_url:
    print("=== Testing remote endpoint ===")
    try:
        import urllib.request, json
        req = urllib.request.Request(
            f"{tunnel_url}/api/generate",
            data=json.dumps({
                'model': 'hp-1000',
                'prompt': 'Say hi in one word',
                'stream': False
            }).encode(),
            headers={'Content-Type': 'application/json'}
        )
        resp = urllib.request.urlopen(req, timeout=60)
        data = json.loads(resp.read())
        print(f"Response: {data.get('response', 'no response field')}")
        print("\nREMOTE HP-1000 IS WORKING!")
    except Exception as e:
        print(f"Test failed (model might still be loading): {e}")
        print("Wait 30s and re-run this cell")
else:
    print("No tunnel URL — go back and fix Cell 4")

In [ ]:
# Cell 6: Keep alive — prevents Kaggle idle shutdown (30 min default)
# Run this cell LAST and leave it running
import time, threading

def keep_alive():
    while True:
        try:
            # Ping the model to keep it loaded in VRAM
            urllib.request.urlopen(
                urllib.request.Request(
                    'http://localhost:11434/api/generate',
                    data=json.dumps({'model': 'hp-1000', 'prompt': '', 'stream': False, 'keep_alive': '5m'}).encode(),
                    headers={'Content-Type': 'application/json'}
                ),
                timeout=30
            )
        except:
            pass
        time.sleep(60)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("Keep-alive thread started — hp-1000 stays loaded in VRAM")
print(f"Tunnel URL: {tunnel_url}")
print("\n>>> This cell runs forever. Do NOT stop it. <<<")
print(">>> When done, stop the notebook manually. <<<")